<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Visual inspection with few defects

**[Visual inspection with few defects](https://learning.nextia-ai.com/courses/vision/m09/visual-inspection-with-few-defects/)** · Vision AI: See, Count and Inspect · Case study 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** inspect two kinds of circuit board from the open VisA photos. You compare four inspectors on the same photos and under one measure: a simple baseline, a supervised classifier that learns from only 10 damaged photos, the course's small PatchCore that learns from good photos only, and a CLIP prompt with no labels. You choose the pass, check and fail lines from an inspection budget on the validation photos, measure once on the test photos, look at the heat maps, and run the chosen inspector on new boards.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 3 to 5 minutes on a laptop CPU, plus the downloads (the photos 37 MB, ResNet-18 47 MB, CLIP 605 MB). |
| **Needs** | An internet connection for the setup cells and the two models. No account and no key. |
| **You should know** | Module 6 of this course: [Two ways to find damage](https://learning.nextia-ai.com/courses/vision/m06/two-ways-to-find-damage/), [Anomaly-based inspection](https://learning.nextia-ai.com/courses/vision/m06/anomaly-based-inspection/) and [Set the line with the warehouse](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1, matplotlib 3.11.2, PyTorch 2.15.0, timm 1.0.31 and open_clip 4.2.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*). Colab, Kaggle, Windows and Linux are not tested |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m09/visual-inspection-with-few-defects/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M09-L01-visual-inspection-with-few-defects/visual-inspection-with-few-defects-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, open-clip-torch 4.2.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31 open-clip-torch==4.2.1
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)
import open_clip; print("open-clip-torch", open_clip.__version__)

## Setup: get the board photos

Run the next cell. It downloads a fixed sample of the VisA inspection photos (about 37 MB, one zip file) and the course's helper `compare.py` into a folder `visa-boards`. It checks each file's checksum and unpacks the zip. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `visa-boards` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the [VisA dataset](https://github.com/amazon-science/spot-diff) by Zou, Jeong, Pemula, Zhang and Dabeer (Amazon), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Nextia Learning took two of its twelve classes (the boards `pcb1` and `pcb4`), chose 800 photos of each, and resized them to 320 pixels wide. The file `dataset.md` next to this notebook lists every change.

In [ ]:
# Setup: download the board photos and compare.py into visa-boards/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "visa-pcb.zip": "54563c3066fb6485e2e2801895f143dd63836f49ea8293db50b7630c0cd8181d",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("visa-boards").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + ("M09-L01-visual-inspection-with-few-defects/" if name.endswith(".zip") else "") + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder visa-boards and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in visa-boards/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation and the cost table

Fernhill Circuits (a made-up company) builds two kinds of circuit board. An inspector, a person, can check 100 boards in every 1,000. Ayumi, the quality lead, put a price on each mistake. These are **assumptions**, not measured facts:

- a faulty board that is shipped costs 60 dollars (the customer's return, a new board, shipping);
- a good board that is scrapped costs 15 dollars (its parts and the assembly time);
- one check by the inspector costs 0.50 dollars (one minute at 30 dollars an hour).

The course's `compare.py` turns these costs into a cost per 1,000 boards. Its key names come from the returns warehouse of the course ("damaged sold as new"); here they mean "faulty board shipped".

In [ ]:
# The cost table of the board maker (assumptions, not measured): per 1,000 boards of one type.
# compare.py calls a board that fails "damaged"; its key names come from the course's returns warehouse.
COSTS = {
    "damaged_sold_as_new": 60.0,   # a faulty board shipped: the customer's return, a new board, shipping
    "good_marked_damaged": 15.0,   # a good board scrapped: its parts and the assembly time
    "check_by_a_person": 0.5,      # one minute of an inspector at 30 dollars an hour
    "checks_per_day": 100,         # the inspector's budget: 100 checks per 1,000 boards
    "returns_per_day": 1000,       # costs are given per 1,000 boards
}
print(COSTS)

## 2. The data and the first checks

Each row of the manifest is one photo. The columns that matter: `part` (the board type: VisA's `pcb1` or `pcb4`), `split`, `status` (good or damaged), `damage` (VisA's damage types, separated by `; `) and `mask_file` (the true damage area, for damaged photos).

The splits: `train` (200 good and only 10 damaged photos per board), `val` (to choose methods and lines), `test` (opened once, at the end), `new` (a batch of new boards for the end-to-end run), and `extra` (20 more damaged photos, kept back for one experiment).

In [ ]:
boards = pd.read_csv("data/visa-pcb/manifest.csv")
PARTS = ["pcb1", "pcb4"]
SPLITS = ["train", "val", "test", "new", "extra"]
print(boards.groupby(["part", "split", "status"]).size().unstack(fill_value=0))

> **Check.** You should see the same counts for both boards: train 200 + 10, val 100 + 20, test 300 + 40, new 100 + 10, extra 20 damaged.

Two more checks: no photo is in the sample twice (the checksum of each source file), and the photos have one size per board type.

In [ ]:
print("photos:", len(boards), "| different source files:", boards["source_sha256"].nunique())
print(boards.groupby("part")[["source_width", "source_height"]].agg(["min", "max"]))
sizes = {Image.open("data/visa-pcb/" + f).size for f in boards["file"]}
print("sizes in the sample:", sorted(sizes))

Now look at the boards. The next cell shows two good and two damaged photos of each board, with the damage type as the title.

In [ ]:
for part in PARTS:
    rows = boards[(boards["part"] == part) & (boards["split"] == "train")]
    pick = pd.concat([rows[rows["status"] == "good"].head(2), rows[rows["status"] == "damaged"].head(2)])
    show(["data/visa-pcb/" + f for f in pick["file"]], [d if d == d else "good" for d in pick["damage"]], size=2.6)

> **Predict.** Board 1 photos look almost the same from photo to photo. Board 4 photos are less regular. Which board will be easier for a method that compares each photo with the *average* good photo?

## 3. The comparison row

`board_row` scores one method on one board type the way `compare.inspection_row` does: it chooses the cheapest pass, check and fail lines on the **validation** photos, inside the budget of 100 checks per 1,000 boards, and measures the cost on the **test** photos. It also finds the damage type that passes most often (the worst case).

In [ ]:
def board_row(method, part, scores, costs=COSTS):
    """The comparison-table row of one method on one board type: lines chosen on validation,
    cost measured on test, as compare.inspection_row does with the board maker's costs."""
    rows = boards[boards["part"] == part]
    val, test = rows[rows["split"] == "val"], rows[rows["split"] == "test"]
    sv, dv = [scores[i] for i in val["item_id"]], (val["status"] == "damaged").values
    st, dt = [scores[i] for i in test["item_id"]], (test["status"] == "damaged").values
    lines = cp.choose_lines(sv, dv, costs)
    c = cp.inspection_costs(st, dt, lines["check_line"], lines["fail_line"], costs)
    z = cp.zones(st, lines["check_line"], lines["fail_line"])
    found = {}
    for zone, damage, d in zip(z, test["damage"].fillna(""), dt):
        if d:
            for one in damage.split("; "):
                found.setdefault(one, []).append(zone != "pass")
    worst = min(found, key=lambda k: (np.mean(found[k]), k))
    return {"task": "inspect", "method": method, "setting": f"VisA {part} test", "metric": "cost per 1,000 boards",
            "value": c["cost_per_day"], "worst": worst, "worst_value": round(float(np.mean(found[worst])), 3),
            "label": "measured", "auroc": cp.roc_auc(st, dt), "lines": [lines["check_line"], lines["fail_line"]],
            "validation_cost": lines["cost_per_day"], "test": c}

## 4. A baseline: difference from the average good board

Before any model, a rule. Make every photo small and grey, take the average of the good training photos, and score each photo by how far it is from that average.

In [ ]:
def small_grey(path):
    """The photo as a small grey picture: 64 x 48 numbers from 0 to 1."""
    return np.asarray(Image.open(path).convert("L").resize((64, 48), Image.BILINEAR), np.float32) / 255

def average_difference_scores(part):
    """Baseline: how far each photo is from the average good board (mean absolute difference)."""
    rows = boards[boards["part"] == part]
    good = rows[(rows["split"] == "train") & (rows["status"] == "good")]
    average = np.mean([small_grey("data/visa-pcb/" + f) for f in good["file"]], axis=0)
    return {i: float(np.abs(small_grey("data/visa-pcb/" + f) - average).mean())
            for i, f in zip(rows["item_id"], rows["file"])}

base = {part: average_difference_scores(part) for part in PARTS}
for part in PARTS:
    test = boards[(boards["part"] == part) & (boards["split"] == "test")]
    print(part, "AUROC on test:", cp.roc_auc([base[part][i] for i in test["item_id"]], test["status"] == "damaged"))

> **Check.** You should see 0.916 for board 1 and 0.531 for board 4. Board 1 photos are lined up, so the average is a sharp picture of a good board. Board 4 photos move, so the average is a blur.

## 5. Features from ResNet-18

The supervised classifier and PatchCore both use ResNet-18 (Module 6). One pass through the network gives two views of each photo: 784 **patches** of 384 numbers (for PatchCore) and one **embedding** of 512 numbers (for the classifier). This cell downloads ResNet-18 (47 MB) and reads all 1,600 photos in about 30 seconds.

In [ ]:
import torch, timm
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
torch.manual_seed(0)
torch.set_num_threads(4)
# ResNet-18 trained on ImageNet (timm, Apache-2.0 weights), at a fixed revision, as in Module 6.
weights = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
backbone = timm.create_model("resnet18.a1_in1k", pretrained=True, features_only=True, out_indices=(2, 3, 4),
                             pretrained_cfg_overlay={"file": weights}).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def features(path):
    """Two views of one photo at 224 x 224: 784 patches of 384 numbers (layers 2 and 3),
    and one embedding of 512 numbers for the whole photo (layer 4, averaged)."""
    a = np.asarray(Image.open(path).convert("RGB").resize((224, 224), Image.BILINEAR), np.float32) / 255
    x = ((torch.from_numpy(a).permute(2, 0, 1) - MEAN) / STD)[None]
    with torch.no_grad():
        f2, f3, f4 = backbone(x)
        f2 = F.avg_pool2d(f2, 3, 1, 1)
        f3 = F.interpolate(F.avg_pool2d(f3, 3, 1, 1), size=f2.shape[-2:], mode="bilinear", align_corners=False)
        p = torch.cat([f2, f3], 1)[0]
    return p.permute(1, 2, 0).reshape(-1, p.shape[0]).numpy(), f4.mean((2, 3))[0].numpy()

PATCHES, EMBEDDING = {}, {}
for item, f in zip(boards["item_id"], boards["file"]):
    PATCHES[item], EMBEDDING[item] = features("data/visa-pcb/" + f)
print(f"{len(PATCHES)} photos: {PATCHES[item].shape[0]} patches and one embedding of {EMBEDDING[item].shape[0]} numbers each")

## 6. Supervised: learn from 10 damaged photos

A logistic regression learns good against damaged from the embeddings of 200 good and **10 damaged** training photos of each board. `class_weight="balanced"` makes the 10 damaged photos count as much as the 200 good ones.

In [ ]:
from sklearn.linear_model import LogisticRegression

def supervised_scores(part, train_splits=("train",)):
    """Supervised: logistic regression on the ResNet-18 embeddings of the labelled training photos."""
    rows = boards[boards["part"] == part]
    tr = rows[rows["split"].isin(train_splits)]
    X = np.stack([EMBEDDING[i] for i in tr["item_id"]])
    model = LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced").fit(X, tr["status"] == "damaged")
    p = model.predict_proba(np.stack([EMBEDDING[i] for i in rows["item_id"]]))[:, 1]
    return dict(zip(rows["item_id"], p.astype(float)))

sup = {part: supervised_scores(part) for part in PARTS}

## 7. Anomaly-based: the small PatchCore

PatchCore learns from good photos only. It keeps a memory of 20,000 patches of the 200 good training photos of each board. A photo's score is the distance of its oddest patch to the nearest good patch. It also gives a 28 x 28 heat map. This takes about 20 seconds.

In [ ]:
def patchcore(part, memory_size=20000):
    """The course's small PatchCore: a memory of good patches; a photo's score is its largest patch distance.
    Returns the scores and a 28 x 28 heat map for every photo of this board type."""
    rows = boards[boards["part"] == part]
    good = rows[(rows["split"] == "train") & (rows["status"] == "good")]
    bank = np.vstack([PATCHES[i] for i in good["item_id"]])
    rng = np.random.default_rng(0)
    memory = bank[rng.choice(len(bank), memory_size, replace=False)].astype(np.float32)
    memory_sq = (memory ** 2).sum(1)
    scores, maps = {}, {}
    for i in rows["item_id"]:
        q = PATCHES[i]
        d = np.sqrt(np.maximum(((q ** 2).sum(1)[:, None] + memory_sq[None] - 2 * q @ memory.T).min(1), 0))
        scores[i], maps[i] = float(d.max()), d.reshape(28, 28)
    return scores, maps

pc, pc_maps = {}, {}
for part in PARTS:
    pc[part], maps = patchcore(part)
    pc_maps.update(maps)

The heat maps of two damaged test boards, one of each type, next to the true damage:

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(9, 5))
for k, part in enumerate(PARTS):
    one = boards[(boards["part"] == part) & (boards["split"] == "test") & (boards["status"] == "damaged")].iloc[0]
    ax[k, 0].imshow(Image.open("data/visa-pcb/" + one["file"])); ax[k, 0].set_title(f"{part}: {one['damage']}", fontsize=9)
    ax[k, 1].imshow(Image.open("data/visa-pcb/" + one["mask_file"])); ax[k, 1].set_title("true damage (mask)", fontsize=9)
    ax[k, 2].imshow(pc_maps[one["item_id"]], cmap="inferno"); ax[k, 2].set_title("PatchCore heat map", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.show()

## 8. Zero-shot: a CLIP prompt, no labels

CLIP (Module 3) compares a photo with texts. We write two sets of texts, "a photo of a flawless circuit board" and "a photo of a damaged circuit board" (four words and four templates each), and score each photo by the share of "damaged". No photo of the boards is used to learn. This cell downloads CLIP ViT-B/32 (605 MB) and reads the 1,600 photos in about 45 seconds.

> **Predict.** CLIP gave an AUROC of 0.808 on the course's candles. Will it do better or worse on circuit boards?

In [ ]:
import open_clip
# CLIP ViT-B/32 trained by LAION (MIT licence), at a fixed revision, as in Module 3.
clip_weights = hf_hub_download("laion/CLIP-ViT-B-32-laion2B-s34B-b79K", "open_clip_model.safetensors",
                               revision="1a25a446712ba5ee05982a381eed697ef9b435cf")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained=clip_weights)
clip_model.eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")
GOOD_WORDS = ["flawless", "perfect", "intact", "good"]
BAD_WORDS = ["damaged", "defective", "broken", "scratched"]
TEMPLATES = ["a photo of a {} circuit board.", "a close-up photo of a {} printed circuit board.",
             "a photo of a {} electronic board.", "an industrial photo of a {} circuit board."]

def text_vector(words):
    with torch.no_grad():
        e = clip_model.encode_text(tokenizer([t.format(w) for w in words for t in TEMPLATES]))
        e = e / e.norm(dim=-1, keepdim=True)
        v = e.mean(0)
        return (v / v.norm()).numpy()

TEXT = np.stack([text_vector(GOOD_WORDS), text_vector(BAD_WORDS)])
CLIP_IMAGE = {}
with torch.no_grad():
    for item, f in zip(boards["item_id"], boards["file"]):
        e = clip_model.encode_image(clip_preprocess(Image.open("data/visa-pcb/" + f).convert("RGB"))[None])[0]
        CLIP_IMAGE[item] = (e / e.norm()).numpy()

def clip_scores(part):
    """Zero-shot: the share of 'damaged' between the two texts (softmax of 100 x cosine similarity)."""
    rows = boards[boards["part"] == part]
    sims = 100 * np.stack([CLIP_IMAGE[i] for i in rows["item_id"]]) @ TEXT.T
    p = np.exp(sims[:, 1]) / np.exp(sims).sum(1)
    return dict(zip(rows["item_id"], p.astype(float)))

clip = {part: clip_scores(part) for part in PARTS}

## 9. A fair comparison: validation first

Every method now has a score for every photo. First we compare them on the **validation** photos only: the cheapest lines for 100 checks per 1,000 boards, and their cost there.

In [ ]:
METHODS = {"baseline": base, "supervised": sup, "patchcore": pc, "clip": clip}
rows = [board_row(name, part, scores[part]) | {"part": part} for name, scores in METHODS.items() for part in PARTS]
validation = pd.DataFrame([{"method": r["method"], "part": r["part"], "validation cost": r["validation_cost"]} for r in rows])
print(validation.pivot(index="method", columns="part", values="validation cost").assign(mean=lambda t: t.mean(axis=1)).round(0).astype(int))

> **Check.** You should see the supervised classifier as the cheapest on validation (mean 1229 dollars per 1,000 boards). We choose it now, **before** we look at the test photos.

A common mistake: CLIP's "damaged" share looks like a probability, so a line at 0.5 looks natural. It is not: every CLIP score is far below 0.5, so a line at 0.5 passes every board.

In [ ]:
test1 = boards[(boards["part"] == "pcb1") & (boards["split"] == "test")]
s = [clip["pcb1"][i] for i in test1["item_id"]]
print(f"CLIP scores on board 1 from {min(s):.3f} to {max(s):.3f}")
print(cp.inspection_costs(s, test1["status"] == "damaged", 0.5, 0.5, COSTS)["cost_per_day"], "dollars per 1,000 boards")

## 10. The test photos, once

Now we open the test photos for every method, with the lines from validation. The reference rows: no inspection (every faulty board is shipped), the inspector checks 100 random boards in 1,000, and a perfect inspector that checks only the damaged boards.

In [ ]:
test_table = pd.DataFrame([{"method": r["method"], "part": r["part"], "AUROC": r["auroc"], "test cost": r["value"],
                            "checks per 1,000": r["test"]["checks_per_day"], "faulty shipped": r["test"]["missed"],
                            "good scrapped": r["test"]["good_failed"], "worst type": f"{r['worst']} {r['worst_value']}"}
                           for r in rows])
print(test_table.to_string(index=False))
print("mean test cost:", test_table.groupby("method")["test cost"].mean().round(0).astype(int).to_dict())

> **Check.** Mean test cost per 1,000 boards, in whole dollars: supervised 1632, PatchCore 2510. No inspection costs 7,059.

The rows also go into the course's comparison table, with the `inspect` task:

In [ ]:
table = cp.Table()
for r in rows:
    table.add({k: r.get(k) for k in cp.COLUMNS} | {"setting": f"VisA {r['part']} test", "ms_per_image": None,
                                                    "model_mb": None, "labelled_images": None, "licence": None})
table.show(task="inspect", columns=("method", "setting", "value", "worst", "worst_value"))

## 11. End to end: a batch of new boards

The decision: the supervised classifier with the lines from validation. Here it meets 110 new boards of each type that no step has seen. Each board goes to pass, check or fail.

In [ ]:
for part in PARTS:
    r = next(x for x in rows if x["method"] == "supervised" and x["part"] == part)
    new = boards[(boards["part"] == part) & (boards["split"] == "new")]
    zone = cp.zones([sup[part][i] for i in new["item_id"]], *r["lines"])
    print(part, f"lines {r['lines'][0]:.3f} and {r['lines'][1]:.3f}:",
          pd.crosstab(new["status"].values, zone).to_dict("index"))
    missed = new[(zone == "pass") & (new["status"] == "damaged").values]
    if len(missed):
        show(["data/visa-pcb/" + f for f in missed["file"]], [f"passed: {d}" for d in missed["damage"]], size=2.6)

## 12. Your turn

> **Your turn 1.** If nobody inspects board 1, every damaged test board is shipped. Compute the cost per 1,000 boards in `no_inspection`: the share of damaged boards in the test photos, times 1,000, times the cost of a faulty board shipped. Round to one decimal.

In [ ]:
no_inspection = ...  # the cost per 1,000 boards of board 1 if nobody inspects: every damaged test board is shipped

In [ ]:
expect('no_inspection', no_inspection, 7058.8, tolerance=0.001)

> **Your turn 2.** Ayumi's inspector may be ill. With only 50 checks per 1,000 boards, what does the supervised classifier cost? Make a copy of `COSTS` with `checks_per_day=50`, call `board_row` for each board with it, and put the mean test cost in `budget_50` (rounded to one decimal). Predict first: a little more than 1632 dollars, or a lot more?

In [ ]:
budget_50 = ...  # the mean cost per 1,000 boards of the two board types, with lines chosen for 50 checks

In [ ]:
expect('budget_50', budget_50, 2033.0, tolerance=0.001)

> **Your turn 3.** What if Ayumi's team labels 20 more damaged photos of each board? Train the supervised classifier on the splits `train` and `extra` (`supervised_scores(part, ("train", "extra"))`), and put the mean test cost of the two boards in `more_labels`, rounded to one decimal. This is an experiment to plan the next step. The method was chosen before it, so it does not change the choice.

In [ ]:
more_labels = ...  # the mean cost per 1,000 boards when the supervised model learns from 30 damaged photos

In [ ]:
expect('more_labels', more_labels, 783.8, tolerance=0.02)

## Next

The [case study page](https://learning.nextia-ai.com/courses/vision/m09/visual-inspection-with-few-defects/) explains each result, the decision for Fernhill Circuits, the limits, responsible use, and how to make this a portfolio piece. It also has the knowledge checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Classify products and pets from photos](https://learning.nextia-ai.com/courses/vision/m09/classify-products-and-pets-from-photos/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m09/visual-inspection-with-few-defects/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are a sample of the VisA dataset by Zou, Jeong, Pemula, Zhang and Dabeer (Amazon), CC BY 4.0, resized by Nextia Learning; no photo shows a person.